# Kiểm tra kích thước cho cả 3 loại — không train lại

Dùng nguyên ZIP `fruit_all3_v2_20261116.zip` đã có trên My Drive và kết quả train hiện tại.
Không cần upload ZIP/model lần nữa. Chọn T4 GPU; đảm bảo lượt train trước đã dừng.

Notebook chỉ đo validation297 ảnh ở 640 và 1024, so baseline/trial_v1/trial_v2.
Kết quả1280 dùng lại từ báo cáo đã gửi. Không chạy test, không thay model app.


In [ ]:
import torch
assert torch.cuda.is_available(), 'Chọn T4 GPU trong Runtime → Change runtime type. Không chạy lượt này bằng CPU.'
print('GPU:', torch.cuda.get_device_name(0), '| Torch:', torch.__version__)


## 1. Kết nối Drive và kiểm tra dữ liệu cũ


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import hashlib, json, shutil, zipfile

ZIP_DRIVE = Path('/content/drive/MyDrive/fruit_all3_v2_20261116.zip')
ZIP_LOCAL = Path('/content/fruit_all3_v2_20261116.zip')
ROOT = Path('/content/fruit_all3_v2_20261116')
PROJECT = Path('/content/drive/MyDrive/fruit_all3_v2_20261116/runs')
EXPECTED_SHA256 = '20d8de7c2b47c3d47df75f65907645caea2ce40ae358b064751334611c739550'

def sha_file(path):
    value = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(4 * 1024 * 1024), b''):
            value.update(chunk)
    return value.hexdigest()

assert ZIP_DRIVE.is_file(), 'Upload đúng tên ZIP vào thư mục gốc My Drive của tài khoản vừa mount.'
if not ZIP_LOCAL.exists():
    print('Copy ZIP từ Drive...')
    shutil.copyfile(ZIP_DRIVE, ZIP_LOCAL)
assert sha_file(ZIP_LOCAL) == EXPECTED_SHA256, 'ZIP sai phiên bản hoặc chưa upload đủ.'
ROOT.mkdir(exist_ok=True, parents=True)
with zipfile.ZipFile(ZIP_LOCAL) as archive:
    assert all((ROOT / item.filename).resolve().is_relative_to(ROOT.resolve()) for item in archive.infolist())
    archive.extractall(ROOT)
manifest = json.loads((ROOT / 'manifest.json').read_text(encoding='utf-8'))
for record in manifest['files']:
    assert sha_file(ROOT / record['image']) == record['image_sha256'], record['image']
    assert sha_file(ROOT / record['label']) == record['label_sha256'], record['label']
for record in manifest['models'].values():
    assert sha_file(ROOT / record['path']) == record['sha256']
print(json.dumps(manifest['summary'], ensure_ascii=False, indent=2))
print('Kết quả lưu trên Drive:', PROJECT)


## 2. Thư viện (giữ nguyên Torch/CUDA)


In [ ]:
import importlib.metadata as metadata
import subprocess, sys
from packaging.requirements import Requirement

torch_before = {name: metadata.version(name) for name in ('torch', 'torchvision')}
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps',
                'ultralytics==8.4.180', 'ultralytics-thop', 'polars', 'nvidia-ml-py'], check=True)
# Polars separates its native runtime into an explicit matching-version wheel.
runtime_requirements = []
for requirement in metadata.requires('polars') or []:
    parsed = Requirement(requirement)
    if parsed.name.startswith('polars-runtime-') and (parsed.marker is None or parsed.marker.evaluate({'extra': ''})):
        runtime_requirements.append(parsed.name + str(parsed.specifier))
if runtime_requirements:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps', *runtime_requirements], check=True)
assert torch_before == {name: metadata.version(name) for name in torch_before}, 'Torch/torchvision changed unexpectedly'
import ultralytics, polars
print('Ultralytics:', ultralytics.__version__, '| Polars:', polars.__version__)


## 3. Cấu hình

Giữ đúng NAME của lượt train v2. BATCH=2 như báo cáo vừa gửi.
Không có ACTION=train trong notebook này.


In [ ]:
NAME = 'all3_balanced_v2'
BATCH = 2
assert (PROJECT / NAME / 'weights/best.pt').is_file(), 'Không tìm thấy best.pt; kiểm tra NAME và tài khoản Drive.'
assert (PROJECT / NAME / 'comparison_val.json').is_file(), 'Cần giữ comparison_val.json của lượt vừa chạy.'
print('Model đã train:', PROJECT / NAME / 'weights/best.pt')


## 4. Đo 640/1024

Chạy một lượt; log lưu Drive. Nếu Colab ngắt, chạy lại setup và ô này:
các model/kích thước đã đo xong sẽ được dùng lại, không đo lại toàn bộ.


In [ ]:
RUNNER_SOURCE = '"""Validation-only comparison at 640/1024; reuse existing 1280 report. No training."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport time\n\nEXPECTED_V2 = "d0abaa79cc3784849dc3bf7c178033c56df2990b71a2f69397dfb94fc463796d"\n\n\ndef sha(path):\n    return hashlib.sha256(path.read_bytes()).hexdigest()\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--root", type=Path, required=True)\n    parser.add_argument("--project", type=Path, required=True)\n    parser.add_argument("--name", default="all3_balanced_v2")\n    parser.add_argument("--batch", type=int, default=2)\n    args = parser.parse_args()\n    if not args.name or any(c not in "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-" for c in args.name):\n        raise ValueError("Invalid NAME")\n    if not 1 <= args.batch <= 8:\n        raise ValueError("Invalid BATCH")\n    import torch\n    from ultralytics import YOLO\n    if not torch.cuda.is_available():\n        raise RuntimeError("Select T4 GPU in Colab. No full validation on laptop CPU.")\n    from metrics_all3 import evaluate_counts\n    root = args.root.resolve()\n    run = args.project / args.name\n    manifest = json.loads((root / "manifest.json").read_text(encoding="utf-8"))\n    prior_path = run / "comparison_val.json"\n    prior = json.loads(prior_path.read_text(encoding="utf-8"))\n    manifest_hash = sha(root / "manifest.json")\n    assert prior["protocol"]["manifest_sha256"] == manifest_hash\n    assert prior["protocol"]["imgsz"] == 1280 and prior["protocol"]["split"] == "val"\n    assert prior["protocol"]["AP_conf"] == .001 and prior["protocol"]["count_conf"] == .4\n    assert prior["protocol"]["iou"] == .7 and prior["protocol"]["agnostic_nms"] is True\n    paths = {name: root / info["path"] for name, info in manifest["models"].items()}\n    paths["trial_v2"] = run / "weights/best.pt"\n    hashes = {name: sha(path) for name, path in paths.items()}\n    assert hashes == prior["protocol"]["models"], "Model changed since the submitted report"\n    assert hashes["trial_v2"] == EXPECTED_V2, "Not the candidate from the submitted comparison"\n    output = run / f"resolution_check_{EXPECTED_V2[:8]}_batch{args.batch}"\n    output.mkdir(parents=True, exist_ok=True)\n    progress = output / "progress.json"\n    selected = [r for r in manifest["files"] if r["split"] == "val"]\n    assert len(selected) == 297\n    rows = []\n    for old in prior["results"]:\n        rows.append({"imgsz": 1280, "batch": None, "source": "existing submitted validation; original batch was not recorded in protocol",\n                     **{key: old[key] for key in ("model", "sha256", "mAP50", "mAP50_95", "AP50", "operating_point")}})\n    for size in (640, 1024):\n        for name, path in paths.items():\n            cache_dir = output / f"imgsz_{size}"\n            cache_dir.mkdir(exist_ok=True)\n            cache = cache_dir / (name + ".json")\n            receipt = {"imgsz": size, "batch": args.batch, "models": hashes,\n                       "manifest_sha256": manifest_hash, "AP_conf": .001, "count_conf": .4,\n                       "iou": .7, "agnostic_nms": True}\n            if cache.exists():\n                row = json.loads(cache.read_text(encoding="utf-8"))\n                assert row["receipt"] == receipt, "Existing cache uses other settings"\n                rows.append(row)\n                continue\n            def status(stage):\n                progress.write_text(json.dumps({"stage": f"{name} imgsz{size}: {stage}", "time": time.time()}), encoding="utf-8")\n            status("AP evaluation")\n            model = YOLO(str(path))\n            metrics = model.val(data=str(root / "data.yaml"), split="val", imgsz=size,\n                                conf=.001, iou=.7, agnostic_nms=True, batch=args.batch, device=0,\n                                project=str(cache_dir), name=name + "_AP", exist_ok=True, plots=False)\n            status("per-class F1 and counting")\n            counts = evaluate_counts(model, root, selected, size, args.batch)\n            row = {"imgsz": size, "batch": args.batch, "source": "new validation; no training", "receipt": receipt,\n                   "model": name, "sha256": hashes[name], "mAP50": float(metrics.box.map50),\n                   "mAP50_95": float(metrics.box.map),\n                   "AP50": {model.names[int(c)]: float(v) for c, v in zip(metrics.box.ap_class_index, metrics.box.ap50)},\n                   "operating_point": counts["summary"]}\n            (cache_dir / (name + "_per_image.json")).write_text(json.dumps(counts, indent=2), encoding="utf-8")\n            cache.write_text(json.dumps(row, indent=2), encoding="utf-8")\n            rows.append(row)\n            del model\n            torch.cuda.empty_cache()\n    rows.sort(key=lambda r: (r["imgsz"], list(paths).index(r["model"])))\n    result = {"split": "val", "manifest_sha256": manifest_hash, "models": hashes, "results": rows,\n              "limitations": manifest["limitations"] + [\n                  "1280 rows reused from the submitted run; batch was not recorded there. Only 640/1024 newly evaluated with the same explicit batch.",\n                  "Resolution selection uses development validation, not legacy test or an independent final benchmark.",\n                  "No model training, app changes, CLIP or teacher evaluation in this check."]}\n    target = output / "resolution_comparison.json"\n    target.write_text(json.dumps(result, indent=2), encoding="utf-8")\n    lines = ["# Validation by model and image size", "",\n             "| Size | Model | mAP50 | mAP50-95 | AP banana | AP apple | AP orange | Macro F1 @.4 | Count MAE |",\n             "|---:|---|---:|---:|---:|---:|---:|---:|---:|"]\n    for row in rows:\n        op = row["operating_point"]\n        lines.append(f"| {row[\'imgsz\']} | {row[\'model\']} | {row[\'mAP50\']:.4f} | {row[\'mAP50_95\']:.4f} | " +\n                     " | ".join(f"{row[\'AP50\'].get(k, 0):.4f}" for k in ("banana", "apple", "orange")) +\n                     f" | {op[\'macro_F1\']:.4f} | {op[\'count_MAE_all_classes\']:.3f} |")\n    lines.extend(["", "Legacy labels contain bunch boxes; inspect individual-fruit scenes before selecting a model. Not a full-app benchmark."])\n    target.with_suffix(".md").write_text("\\n".join(lines), encoding="utf-8")\n    progress.write_text(json.dumps({"stage": "completed", "time": time.time()}), encoding="utf-8")\n    print("\\n".join(lines), flush=True)\n    print("Report:", target, flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'
(ROOT / 'run_resolution_check.py').write_text(RUNNER_SOURCE, encoding='utf-8')


In [ ]:
import subprocess, sys, signal, time
from collections import deque
from datetime import datetime, timezone
from IPython.display import display, Markdown
output_dir = PROJECT / NAME / f'resolution_check_d0abaa79_batch{BATCH}'
progress = output_dir / 'progress.json'
logs = PROJECT.parent / 'logs'
logs.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
log_path = logs / f'{NAME}_resolution_check_{stamp}.log'
command = [sys.executable, '-u', str(ROOT / 'run_resolution_check.py'),
           '--root', str(ROOT), '--project', str(PROJECT), '--name', NAME, '--batch', str(BATCH)]
print('Log:', log_path, flush=True)
started = time.time()
with log_path.open('w', encoding='utf-8') as output:
    process = subprocess.Popen(command, cwd=ROOT, stdout=output, stderr=subprocess.STDOUT, text=True)
    try:
        while process.poll() is None:
            time.sleep(20)
            stage = 'Đang khởi tạo'
            try:
                stage = json.loads(progress.read_text(encoding='utf-8'))['stage']
            except (OSError, ValueError, KeyError):
                pass
            print(int(time.time() - started), 's |', stage, flush=True)
    except KeyboardInterrupt:
        process.send_signal(signal.SIGINT)
        try:
            process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            process.terminate()
            process.wait(timeout=20)
        raise
    return_code = process.wait()
with log_path.open(encoding='utf-8', errors='replace') as stream:
    print(''.join(deque(stream, maxlen=60)))
if return_code:
    raise RuntimeError(f'Lỗi mã {return_code}; đọc lỗi gốc ngay phía trên. Log: {log_path}')
report = output_dir / 'resolution_comparison.json'
display(Markdown(report.with_suffix('.md').read_text(encoding='utf-8')))
print('Gửi file này:', report)


Gửi `resolution_comparison.json` để chọn model + kích thước dựa trên AP từng loại, F1 và sai số đếm.
Kết quả vẫn dùng validation cũ có nhãn nải; cần kiểm tra ảnh thực và không xem đây là test độc lập.
